# The high-precision hold

What resolution, repeatability and stiffness does a link-rate servo
honestly buy on this drive? Three measurements, no adjectives: the
smallest step the loop resolves, the spread over repeated moves, and
degrees lost per newton-metre of disturbance at two holding currents -
the number a gimbal or a focuser actually cares about.

In [ ]:
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
while not (root / 'host' / 'coaxial').is_dir():
    root = root.parent
sys.path.insert(0, str(root / 'host'))

from coaxial import Coaxial63100

SIMULATED = True                    # False at the bench
PORT = 'COM4'

rig = Coaxial63100(port=PORT, simulated_device=SIMULATED,
                   power_afe=False).open()
rig.drive.source('model')           # the virtual rotor; 'adc' at the bench
rig.gates.arm(bypass_sto=True, ignore_interlock=True)
print(rig)
BLUE, RED, GREY = '#1f4e79', '#c0392b', '#95a5a6'
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.15})


In [ ]:
with rig.motion.servo(amps=3.0, settle=0.3) as s:
    print('step ladder, asked -> got:')
    at = 10.0
    s.to(at, tol=0.3)
    for step in (5.0, 1.0, 0.5, 0.25):
        at += step
        got = s.to(at, tol=0.2, tries=6)
        print('  +%4.2f deg -> %+6.3f (err %+.3f)' % (step, got, s.error))

In [ ]:
reps = []
with rig.motion.servo(amps=3.0, settle=0.3) as s:
    for _ in range(8):
        s.to(0.0, tol=0.4)
        reps.append(s.to(15.0, tol=0.4))
reps = np.array(reps)
print('repeatability at 15 deg over %d moves: mean %.3f, sd %.3f deg'
      % (len(reps), reps.mean(), reps.std()))

In [ ]:
stiff = {}
for amps in (2.0, 4.0):
    with rig.motion.servo(amps=amps, settle=0.3) as s:
        s.to(20.0, tol=0.4)
        sags = []
        loads = (0.02, 0.04, 0.06)
        for load in loads:
            rig.drive.model_param(load=load)
            time.sleep(0.5)
            sags.append(20.0 - s._measure())
        rig.drive.model_param(load=0.0)
        stiff[amps] = np.array(sags)

fig, ax = plt.subplots(figsize=(7, 3.2))
for amps, sags in stiff.items():
    ax.plot(loads, sags, 'o-', label='%.0f A holding' % amps,
            color=BLUE if amps < 3 else RED)
ax.set(xlabel='disturbance N m', ylabel='sag deg (uncorrected)',
       title='stiffness is amperes: sag before the servo corrects')
ax.legend(frameon=False)
fig.tight_layout()
kt = 1.5 * 7 * 0.005
for amps, sags in stiff.items():
    print('%.0f A: %.1f deg/(N m) small-signal - arcsin off %.2f N m held'
          % (amps, np.degrees(1.0 / (7 * kt * amps)), kt * amps))

## The honest envelope

Resolution bottoms out at the sensor (the A1335's 12 bits are 0.09 deg)
and at the ring the mean-measure averages over; repeatability at these
currents is a few tenths; and stiffness is BOUGHT IN AMPERES - double
the holding current, halve the sag, pay it in copper heat the thermal
observer meters (`thermal_budget.ipynb`). Between corrections the drive
is exactly as stiff as a stepper; after one, exactly as true as the
sensor. That division of labour is the whole design.